# Imports

In [ ]:
import os
import sys
import json
import urllib.request
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from datetime import datetime
import requests
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Safely determine the current directory whether in a script or a notebook
try:
    # If running as a standard .py script
    current_dir = Path(__file__).resolve().parent
except NameError:
    # If running in a Jupyter Notebook or interactive environment
    current_dir = Path.cwd()

# Go up one level from 'src' to the project root
project_root = current_dir.parent

print("KG-RAG Pre-Thesis: Data Acquisition")

print(f"Working directory: {Path.cwd()}")
print(f"Project root resolved to: {project_root}")
print()

# Create data directories using the project root as the anchor
data_raw = project_root / "data" / "raw_data"
data_processed = project_root / "data" / "processed_data"
data_raw.mkdir(parents=True, exist_ok=True)
data_processed.mkdir(parents=True, exist_ok=True)

print("Imports successful")
print("Data directories ready")

### MedLayXplain

In [ ]:
try:
    from huggingface_hub import hf_hub_download
    import os
    
    print("Downloading MedLayXPlain validation parquet file directly...")
    print()
    
    hf_token = os.environ.get("HF_TOKEN")
    cache_dir_path = data_raw / "medlayxplain_cache"
    
    # Download ONLY the validation file directly from the repo
    val_file_path = hf_hub_download(
        repo_id="anonymous-medical/MedLayXPlain",
        filename="MedLayXPlain_val.parquet",
        repo_type="dataset",
        token=hf_token,
        cache_dir=str(cache_dir_path)
    )
    
    print("Loading into DataFrame with pandas...")
    df = pd.read_parquet(val_file_path)
    
    medlay_dir = data_raw / "medlayxplain"
    medlay_dir.mkdir(parents=True, exist_ok=True)
    
    output_csv = medlay_dir / "validation_split.csv"
    output_parquet = medlay_dir / "validation_split.parquet"
    
    print(f"Saving to CSV: {output_csv}")
    df.to_csv(output_csv, index=False)
    
    print(f"Saving to Parquet: {output_parquet}")
    df.to_parquet(output_parquet, index=False)
    
    print()
    print("MedLayXPlain Validation Split Ready Successfully")
    print(f"  Samples: {len(df):,}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  CSV Size: {output_csv.stat().st_size / 1024 / 1024:.1f} MB")
    print(f"  Parquet Size: {output_parquet.stat().st_size / 1024 / 1024:.1f} MB")
    
except Exception as e:
    print(f"✗ Error loading MedLayXPlain validation split: {e}")

### UMLS Full Dataset

In [ ]:
import os
import time
import requests
from tqdm import tqdm

# ==========================================
# UMLS Resumable Stream Download
# ==========================================

umls_api_key = os.getenv("UMLS_API_KEY")

if not umls_api_key:
    print("✗ Error: UMLS_API_KEY not found in .env file.")
else:
    target_url = "https://download.nlm.nih.gov/umls/kss/2026AA/umls-2026AA-metathesaurus-full.zip"
    download_api_endpoint = f"https://uts-ws.nlm.nih.gov/download?url={target_url}&apiKey={umls_api_key}"
    
    umls_dir = data_raw / "umls"
    umls_dir.mkdir(parents=True, exist_ok=True)
    output_zip = umls_dir / "umls_2026AA_full.zip"
    
    chunk_size = 1024 * 1024  # 1 MB
    max_retries = 10
    retry_count = 0

    print(f"Target output file: {output_zip}")
    print("Starting/Resuming download...")
    print()

    while retry_count < max_retries:
        # Check how many bytes have already been downloaded to disk
        downloaded_bytes = output_zip.stat().st_size if output_zip.exists() else 0
        
        # Use HTTP Range header to request remaining bytes
        headers = {"Range": f"bytes={downloaded_bytes}-"} if downloaded_bytes > 0 else {}

        try:
            with requests.get(download_api_endpoint, headers=headers, stream=True, timeout=30) as response:
                # 200 = Full file starting fresh, 206 = Partial content resuming
                if response.status_code not in [200, 206]:
                    print(f"✗ Server error status code: {response.status_code}")
                    break

                # Get total content size
                content_length = int(response.headers.get("content-length", 0))
                total_bytes = downloaded_bytes + content_length if response.status_code == 206 else content_length

                mode = "ab" if response.status_code == 206 else "wb"

                with open(output_zip, mode) as f, tqdm(
                    desc="UMLS 2026AA Full",
                    initial=downloaded_bytes,
                    total=total_bytes,
                    unit="iB",
                    unit_scale=True,
                    unit_divisor=1024,
                ) as progress_bar:

                    for chunk in response.iter_content(chunk_size=chunk_size):
                        if chunk:
                            f.write(chunk)
                            progress_bar.update(len(chunk))

            # Check if download complete
            if output_zip.stat().st_size >= total_bytes and total_bytes > 0:
                print()
                print("✓ UMLS Full Release downloaded successfully!")
                print(f"  Final Size: {output_zip.stat().st_size / (1024 ** 3):.2f} GB")
                break

        except (requests.exceptions.RequestException, Exception) as e:
            retry_count += 1
            current_mb = output_zip.stat().st_size / (1024 * 1024) if output_zip.exists() else 0
            print(f"\n⚠ Connection dropped ({e}). Auto-resuming from {current_mb:.1f} MiB in 5s... (Attempt {retry_count}/{max_retries})")
            time.sleep(5)

### MeSH

In [ ]:
import os
import requests
from pathlib import Path
from tqdm import tqdm

# Define output path
MESH_DIR = data_raw / "mesh"
MESH_DIR.mkdir(parents=True, exist_ok=True)
output_xml = MESH_DIR / "desc2026.xml"

url = "https://nlmpubs.nlm.nih.gov/projects/mesh/MESH_FILES/xmlmesh/desc2026.xml"

print("Downloading MeSH Descriptors XML...")
with requests.get(url, stream=True) as response:
    response.raise_for_status()
    total_size = int(response.headers.get('content-length', 0))
    
    with open(output_xml, 'wb') as f, tqdm(
        desc="MeSH XML",
        total=total_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
    ) as progress_bar:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                progress_bar.update(len(chunk))

print("✓ MeSH XML downloaded successfully!")

### README Dataset

In [ ]:
from datasets import load_dataset

# Define the target output directory using the initialized data_raw path
README_DIR = data_raw / "readme"
README_DIR.mkdir(parents=True, exist_ok=True)
output_path = README_DIR / "NoteAid-README"

print("Downloading NoteAid-README dataset from Hugging Face...")

try:
    # Load the dataset directly from the Hugging Face Hub
    dataset = load_dataset("bio-nlp-umass/NoteAid-README")
    
    # Save the dataset splits locally to disk
    dataset.save_to_disk(str(output_path))
    
    print(f"NoteAid-README successfully downloaded and saved to: {output_path}")
    
except Exception as e:
    print(f"Error downloading NoteAid-README dataset: {e}")